# Example 8.1.1 / 1: Match Features, Not Labels

Companion notebook for **8.1.1 Linear, Mixed-Integer and Conic Solvers** in the AMPL lab Documentation, section "Match Features, Not Labels". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Match Features, Not Labels

"Conic" includes distinct cone families, and "quadratic" does not say whether a quadratic term
sits in the objective or in a constraint. HiGHS solves quadratic objectives, but this version of
the steel model adds a quadratic constraint, `Spread`, which bounds the sum of squared
production. Three solvers are asked to solve it; `verbose=False` hides their logs, and the
program prints the status, the profit, whether the solve message mentions a piecewise-linear
approximation (`PLApprox`), and the plan. In `data.py` and `run.py`, `ampl` is an `AMPL` object
that has already read `model.mod`:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set PROD;                                 # products
param rate {PROD} > 0;                    # tons produced per hour
param avail >= 0;                         # hours available
param profit {PROD};                      # profit per ton
param market {PROD} >= 0;                 # most tons that can be sold
var Make {p in PROD} >= 0, <= market[p];  # tons produced, up to the market limit
maximize Total_Profit: sum {p in PROD} profit[p] * Make[p];  # total profit
# production hours fit the hours available
subject to Time: sum {p in PROD} (1 / rate[p]) * Make[p] <= avail;
# a quadratic constraint: squared production is limited
subject to Spread: sum {p in PROD} Make[p]^2 <= 4e7;

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["PROD"] = ["bands", "coils", "plate"]
ampl.param["rate"] = {"bands": 200, "coils": 140, "plate": 160}
ampl.param["profit"] = {"bands": 25, "coils": 30, "plate": 29}
ampl.param["market"] = {"bands": 6000, "coils": 4000, "plate": 3500}
ampl.param["avail"] = 40

In [5]:
# run.py
for solver in ["highs", "cbc", "ipopt"]:
    ampl.solve(solver=solver, verbose=False)
    approx = "PLApprox" in ampl.get_value("solve_message")
    plan = [round(v) for v in ampl.var["Make"].to_dict().values()]
    profit = ampl.obj["Total_Profit"].value()
    print(f"{solver}: {ampl.solve_result}, profit {profit:.0f}, approximated {approx}, Make {plan}")

highs: solved, profit 13282, approximated True, Make [158, 158, 158]
cbc: solved, profit 26563, approximated True, Make [316, 316, 316]
ipopt: solved, profit 196400, approximated False, Make [6000, 0, 1600]


### Pure AMPL: data.dat and model.run

The same in pure AMPL, with the same `model.mod`:

In [6]:
%%writefile data.dat
set PROD := bands coils plate;
param rate := bands 200  coils 140  plate 160;
param profit := bands 25  coils 30  plate 29;
param market := bands 6000  coils 4000  plate 3500;
param avail := 40;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver_msg 0;
option ipopt_options 'print_level=0';
for {s in {"highs", "cbc", "ipopt"}} {
    option solver (s);
    solve;
    printf "%s: %s, profit %.0f\n", s, solve_result, Total_Profit;
}

HiGHS 1.14.0: 
------------ WARNINGS ------------
WARNING.  6 case(s) of "PLApproxDomain_PowConstExp". One of them:
  Argument domain of a 'PowConstExp'
has been reduced to [0.000000, 316.227766] for numerical reasons
(partially controlled by cvt:plapprox:domain.)
WARNING.  6 case(s) of "PLApprox_PowConstExp". One of them:
  An expression of type 'PowConstExp' has been
piecewise-linearly approximated. Set cvt:plapprox:reltol
to control precision (currently 0.010000).
highs: solved, profit 13282
cbc 2.10.12: 
------------ WARNINGS ------------
WARNING.  3 case(s) of "PLApproxDomain_PowConstExp". One of them:
  Argument domain of a 'PowConstExp'
has been reduced to [0.000000, 316.227766] for numerical reasons
(partially controlled by cvt:plapprox:domain.)
WARNING.  3 case(s) of "PLApprox_PowConstExp". One of them:
  An expression of type 'PowConstExp' has been
piecewise-linearly approximated. Set cvt:plapprox:reltol
to control precision (currently 0.010000).
cbc: solved, profit 26563
Ipo